# 🚀 Zara Visual Product Recognition - The "Beast" Pipeline
**Hardware:** 2x Tesla T4 GPUs | **Estrategia:** YOLOv8 Seg + SigLIP 384px + Contrastive Fine-Tuning + Heurísticas Avanzadas

## 🧱 Bloque 1: Setup, Data Hacking & Hard Constraints
En esta fase inicial:
1. Instalamos las dependencias pesadas que usaremos más adelante (`ultralytics`, `timm`, `open_clip_torch`).
2. Extraemos inteligentemente los **SKUs** y **Timestamps** de las URLs de las imágenes.
3. Aplicamos el **Noise Filter**: Eliminamos 53 categorías de productos que son estadísticamente invisibles (perfumes, velas, etc.).
4. Extraemos las **Hard Constraints (Lógica de Secciones)**: Analizamos el `train_df` para aprender qué categorías están permitidas estrictamente en cada sección comercial, bloqueando cruces absurdos (ej. un vestido en la sección de hombre).

In [2]:
# ==========================================
# BLOQUE 1: SETUP, DATA HACKING & HARD CONSTRAINTS
# ==========================================
# 1. Instalación de artillería pesada (silenciada para no ensuciar el notebook)
!pip install -q ultralytics timm open_clip_torch faiss-gpu

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings

# Silenciar warnings
warnings.filterwarnings('ignore')

print("=== 1. INICIANDO DATA ENGINEERING ===")

# 2. Configuración de Rutas en Kaggle
WORK_DIR = Path('/kaggle/working')
IMG_DIR = WORK_DIR / 'images'
IMG_DIR.mkdir(parents=True, exist_ok=True)

# 3. Carga de los Datasets
try:
    bundles_df = pd.read_csv('/kaggle/input/datasets/nicolsaller/inditex/bundles_dataset.csv')
    products_df = pd.read_csv('/kaggle/input/datasets/nicolsaller/inditex/product_dataset.csv')
    train_df = pd.read_csv('/kaggle/input/datasets/nicolsaller/inditex/bundles_product_match_train.csv')
    test_df = pd.read_csv('/kaggle/input/datasets/nicolsaller/inditex/bundles_product_match_test.csv')
    print("✅ CSVs cargados correctamente.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encuentran los datos en {DATA_DIR}. ¡Revisa la ruta en Kaggle!")

# 4. Hacking de Metadatos (Extracción Vectorizada)
print("Extrayendo SKUs y Timestamps...")
sku_pattern = r'/([A-Z0-9]{8,15}(?:-\d+)?)-[pe]'
bundles_df['sku'] = bundles_df['bundle_image_url'].str.extract(sku_pattern, expand=False)
products_df['sku'] = products_df['product_image_url'].str.extract(sku_pattern, expand=False)

ts_pattern = r'ts=(\d+)'
bundles_df['ts'] = bundles_df['bundle_image_url'].str.extract(ts_pattern, expand=False).astype(float)
products_df['ts'] = products_df['product_image_url'].str.extract(ts_pattern, expand=False).astype(float)

# 5. Filtro de Ruido (Noise Filter)
NOISE_CATEGORIES = {
    'BABY ACCESORIES', 'BABY BODY', 'BABY OUTFIT', 'BABY OVERALL', 'BABY PANTY/UNDERP.', 
    'BABY POLO SHIRT', 'BABY PYJAMA', 'BABY ROMPER SUIT', 'BABY SWIMSUIT', 'BATHROBE/DRES.GOWN', 
    'BEACH SANDAL', 'BODY OIL', 'BOOKS', 'BOW TIE/CUMMERBAND', 'CANDLE', 'EAU DE COLOGNE', 
    'EAU DE TOILETTE', 'ENSEMBLE..SET', 'HAIR COSMETICS', 'HAND CREAM', 'HOME SHOES', 'LIP BALM', 
    'LIP SUNSCREEN', 'MATCHES', 'MOISTURISING CREAM', 'NAIL COSMETICS', 'NAIL POLISH', 'NEWBORN', 
    'NEWBORN TRICOT', 'PARKA', 'PERFUME', 'PERFUMED SOAP', 'POWDER BRUSH-PUFF', 'SHAMPOO', 
    'STATIONERY', 'SUSPENDERS', 'TOWEL', 'UMBRELLA', 'UNIFORM', 'BABY LEGGINGS', 'BABY SOCKS', 
    'BABY TRACKSUIT', 'BABY WAISTCOAT', 'BABY WIND-JACKET', 'BIB OVERALL', 'EAU DE PARFUM', 
    'EYES CONTOUR', 'PURSE WALLET', 'SLEEVELESS PAD. JACKET', 'SPORTY SANDAL', 'STOCKINGS-TIGHTS', 
    'UNDERWEAR', 'WALLETS'
}

initial_count = len(products_df)
products_df = products_df[~products_df['product_description'].isin(NOISE_CATEGORIES)].copy()
print(f"🗑️ Ruido eliminado: {initial_count - len(products_df)} productos invisibles descartados.")
print(f"📦 Catálogo final útil: {len(products_df)} productos.")

# 6. Mapeos ultra-rápidos en memoria
bundle_section_map = dict(zip(bundles_df['bundle_asset_id'], bundles_df['bundle_id_section']))
product_desc_map = dict(zip(products_df['product_asset_id'], products_df['product_description']))

# 7. Extracción de Lógica de Negocio (Hard Constraints)
print("\nAprendiendo reglas de cruce de secciones...")
category_allowed_sections = defaultdict(set)

for _, row in train_df.iterrows():
    b_id = row['bundle_asset_id']
    p_id = row['product_asset_id']
    
    sec = bundle_section_map.get(b_id)
    desc = product_desc_map.get(p_id)
    
    if sec and desc and pd.notna(sec) and pd.notna(desc):
        category_allowed_sections[desc].add(sec)

print(f"Reglas aprendidas para {len(category_allowed_sections)} categorías distintas.")

# Función maestra que usaremos al final para aniquilar falsos positivos
def is_valid_for_section(product_id, bundle_id):
    sec = bundle_section_map.get(bundle_id)
    desc = product_desc_map.get(product_id)
    
    if not sec or not desc: return True # Si nos falta info, somos conservadores y lo permitimos
    if desc not in category_allowed_sections: return True # Cold start
    
    return sec in category_allowed_sections[desc]

print("\nBloque 1 Finalizado con éxito. Entorno preparado.")

ERROR: Ignored the following versions that require a different python version: 8.0.10 Requires-Python >=3.7,<=3.11; 8.0.11 Requires-Python >=3.7,<=3.11; 8.0.12 Requires-Python >=3.7,<=3.11; 8.0.13 Requires-Python >=3.7,<=3.11; 8.0.14 Requires-Python >=3.7,<=3.11; 8.0.15 Requires-Python >=3.7,<=3.11; 8.0.16 Requires-Python >=3.7,<=3.11; 8.0.17 Requires-Python >=3.7,<=3.11; 8.0.18 Requires-Python >=3.7,<=3.11; 8.0.19 Requires-Python >=3.7,<=3.11; 8.0.20 Requires-Python >=3.7,<=3.11; 8.0.21 Requires-Python >=3.7,<=3.11; 8.0.22 Requires-Python >=3.7,<=3.11; 8.0.23 Requires-Python >=3.7,<=3.11; 8.0.24 Requires-Python >=3.7,<=3.11; 8.0.25 Requires-Python >=3.7,<=3.11; 8.0.26 Requires-Python >=3.7,<=3.11; 8.0.27 Requires-Python >=3.7,<=3.11; 8.0.28 Requires-Python >=3.7,<=3.11; 8.0.29 Requires-Python >=3.7,<=3.11; 8.0.30 Requires-Python >=3.7,<=3.11; 8.0.31 Requires-Python >=3.7,<=3.11; 8.0.32 Requires-Python >=3.7,<=3.11; 8.0.33 Requires-Python >=3.7,<=3.11; 8.0.34 Requires-Python >=3.7,<=3.

In [3]:
# ==========================================
# SECTION POPULARITY PRIOR
# ==========================================
from collections import Counter
import math

print("Calculando popularidad de productos por sección...")
section_product_freq = defaultdict(Counter)
for _, row in train_df.iterrows():
    sec = bundle_section_map.get(row['bundle_asset_id'])
    if sec:
        section_product_freq[sec][row['product_asset_id']] += 1

section_max_freq = {sec: max(cnt.values()) for sec, cnt in section_product_freq.items()}
print(f"✅ Popularity prior construido para {len(section_product_freq)} secciones.")

# ==========================================
# GROUND TRUTH TRAIN para Bundle-to-Bundle (B2B)
# ==========================================
ground_truth_b2b = defaultdict(set)
for _, row in train_df.iterrows():
    ground_truth_b2b[row['bundle_asset_id']].add(row['product_asset_id'])

print(f"✅ Ground truth B2B construido para {len(ground_truth_b2b)} bundles de train.")
print("\nBloque 1 Finalizado con éxito. Entorno preparado.")

Calculando popularidad de productos por sección...
✅ Popularity prior construido para 3 secciones.
✅ Ground truth B2B construido para 1876 bundles de train.

Bloque 1 Finalizado con éxito. Entorno preparado.


In [5]:
# ==========================================
# BLOQUE 1.5: DESCARGA ASÍNCRONA DE IMÁGENES
# ==========================================
import asyncio
import aiohttp
import nest_asyncio
from tqdm.asyncio import tqdm
from pathlib import Path

nest_asyncio.apply()

print("=== 1.5 DESCARGANDO IMÁGENES AL NUEVO ENTORNO ===")

# Crear directorios
BUNDLES_IMG_DIR = WORK_DIR / 'images/bundles'
PRODUCTS_IMG_DIR = WORK_DIR / 'images/products'
BUNDLES_IMG_DIR.mkdir(parents=True, exist_ok=True)
PRODUCTS_IMG_DIR.mkdir(parents=True, exist_ok=True)

async def download_image(session, url, path, semaphore):
    if path.exists() and path.stat().st_size > 0:
        return True
    try:
        async with semaphore:
            async with session.get(url, timeout=15) as response:
                if response.status == 200:
                    content = await response.read()
                    with open(path, 'wb') as f:
                        f.write(content)
                    return True
                return False
    except:
        return False

async def main_download():
    tasks = []
    semaphore = asyncio.Semaphore(100) # 100 conexiones simultáneas
    
    async with aiohttp.ClientSession() as session:
        # Bundles
        for _, row in bundles_df.iterrows():
            img_path = BUNDLES_IMG_DIR / f"{row['bundle_asset_id']}.jpg"
            tasks.append(download_image(session, row['bundle_image_url'], img_path, semaphore))
            
        # Productos (solo los del catálogo filtrado)
        for _, row in products_df.iterrows():
            img_path = PRODUCTS_IMG_DIR / f"{row['product_asset_id']}.jpg"
            tasks.append(download_image(session, row['product_image_url'], img_path, semaphore))
            
        print(f"Lanzando {len(tasks)} descargas...")
        results = await tqdm.gather(*tasks, desc="Descargando")
        print(f"✅ Exitosos: {sum(r for r in results if r)} | Fallos: {len(tasks) - sum(r for r in results if r)}")

# Ejecutar descarga
asyncio.run(main_download())

=== 1.5 DESCARGANDO IMÁGENES AL NUEVO ENTORNO ===
Lanzando 28370 descargas...


Descargando: 100%|██████████| 28370/28370 [00:23<00:00, 1217.88it/s] 

✅ Exitosos: 28370 | Fallos: 0


## ✂️ Bloque 2: YOLOv8 Segmentation & Spatial Mapping

Para solucionar la "ceguera" del modelo y el ruido del fondo (suelo, paredes), usamos **YOLOv8-Seg**:
1. Detecta la silueta exacta de la persona en el *bundle*.
2. Aplica una máscara que convierte todo el fondo de la imagen en negro puro (RGB 0,0,0).
3. Divide anatómicamente el bounding box de la persona en 4 zonas: `HEAD`, `TORSO`, `LEGS`, `FEET`.
4. Guarda estos recortes limpios en disco junto con la vista `GLOBAL`, etiquetando la zona de la que provienen para aplicar las **Hard Constraints espaciales** después.

In [6]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.0 MB/s eta 0:00:0000:010:01


In [7]:
# ==========================================
# BLOQUE 2: YOLOv8 SEGMENTATION & SPATIAL MAPPING
# ==========================================
from ultralytics import YOLO
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import os
import pandas as pd

print("=== 2. INICIANDO SEGMENTACIÓN CON YOLOv8 ===")

# 1. Crear directorio para guardar los recortes de forma segura y no saturar la RAM
CROPS_DIR = WORK_DIR / 'smart_crops'
CROPS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Cargar el modelo YOLOv8 de segmentación (Usamos 'm' para un balance perfecto entre velocidad y precisión)
# Se descargará automáticamente de los servidores de Ultralytics
yolo_model = YOLO("yolov8x-seg.pt") 

# 3. Preparar lista de bundles a procesar (Train + Test)
# Solo procesaremos los bundles únicos para no repetir trabajo
bundles_to_process = list(set(test_df['bundle_asset_id'].tolist() + train_df['bundle_asset_id'].tolist()))
print(f"Total de bundles a procesar: {len(bundles_to_process)}")

crops_metadata = []

def process_and_save_bundle(bundle_id):
    img_path = str(BUNDLES_IMG_DIR / f"{bundle_id}.jpg") if 'BUNDLES_IMG_DIR' in globals() else str(WORK_DIR / f"images/bundles/{bundle_id}.jpg")
    
    if not os.path.exists(img_path):
        return
    
    img = cv2.imread(img_path)
    if img is None: return
    
    h_img, w_img = img.shape[:2]
    
    # Inferencia (classes=[0] para detectar solo personas)
    results = yolo_model(img, classes=[0], conf=0.3, verbose=False)
    
    # --- 1. Guardar la imagen Global ---
    global_path = CROPS_DIR / f"{bundle_id}_GLOBAL.jpg"
    cv2.imwrite(str(global_path), img)
    crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'GLOBAL', 'crop_path': str(global_path)})
    
    # Procesar detecciones
    for r in results:
        # Si encuentra máscara y bounding box
        if r.masks is not None and r.boxes is not None:
            # Coger la primera persona detectada (la más segura)
            mask = r.masks.data[0].cpu().numpy()
            mask = cv2.resize(mask, (w_img, h_img))
            
            # Remover fondo (lo ponemos a negro)
            img_bg_removed = img.copy()
            img_bg_removed[mask < 0.5] = 0
            
            # Bounding box de la persona
            x1, y1, x2, y2 = map(int, r.boxes.xyxy[0])
            person_crop = img_bg_removed[max(0, y1):min(h_img, y2), max(0, x1):min(w_img, x2)]
            
            if person_crop.size == 0: continue
            
            h_p, w_p = person_crop.shape[:2]
            
            # --- 2. Recortes Anatómicos (Guardamos en disco) ---
            
            # A) HEAD (0% - 20%)
            head = person_crop[0:int(h_p*0.2), 0:w_p]
            head_path = CROPS_DIR / f"{bundle_id}_HEAD.jpg"
            cv2.imwrite(str(head_path), head)
            crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'HEAD', 'crop_path': str(head_path)})
            
            # B) TORSO (20% - 60%)
            torso = person_crop[int(h_p*0.2):int(h_p*0.6), 0:w_p]
            torso_path = CROPS_DIR / f"{bundle_id}_TORSO.jpg"
            cv2.imwrite(str(torso_path), torso)
            crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'TORSO', 'crop_path': str(torso_path)})
            
            # C) LEGS (50% - 85%) - Solapamos un poco con el torso por si las camisetas son largas
            legs = person_crop[int(h_p*0.5):int(h_p*0.85), 0:w_p]
            legs_path = CROPS_DIR / f"{bundle_id}_LEGS.jpg"
            cv2.imwrite(str(legs_path), legs)
            crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'LEGS', 'crop_path': str(legs_path)})
            
            # D) FEET (80% - 100%) - Especial para zapatos, sin suelo ruidoso
            feet = person_crop[int(h_p*0.8):h_p, 0:w_p]
            feet_path = CROPS_DIR / f"{bundle_id}_FEET.jpg"
            cv2.imwrite(str(feet_path), feet)
            crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'FEET', 'crop_path': str(feet_path)})
            
            # Solo procesamos la persona principal para evitar recortes de gente de fondo
            break 

# 4. Ejecutar el procesamiento
for b_id in tqdm(bundles_to_process, desc="Segmentando y Recortando"):
    process_and_save_bundle(b_id)

# 5. Convertir a DataFrame para fácil acceso después
crops_df = pd.DataFrame(crops_metadata)
print(f"\n✅ Segmentación completada. {len(crops_df)} recortes generados y guardados en disco.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
=== 2. INICIANDO SEGMENTACIÓN CON YOLOv8 ===
Total de bundles a procesar: 2331


Segmentando y Recortando:   0%|          | 0/2331 [00:00<?, ?it/s]


✅ Segmentación completada. 11647 recortes generados y guardados en disco.


## 🦅 Bloque 3: SigLIP-384 High-Res Embeddings

Reemplazamos la visión estándar por **SigLIP-SO400M a 384px**. 
1. Cargamos el modelo en DataParallel usando las 2 GPUs T4 para destruir los tiempos de inferencia.
2. Pasamos los ~26.000 productos del catálogo.
3. Pasamos los 11.655 recortes anatómicos de los *bundles* (GLOBAL, HEAD, TORSO, LEGS, FEET).
4. Guardamos todo en diccionarios anidados para que la búsqueda posterior sea instantánea.

In [8]:
!pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00


In [10]:
# ==========================================
# BLOQUE 3: HIGH-RES EMBEDDINGS (SigLIP2 384px)
# ==========================================
import torch
import open_clip
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import os

print("=== 3. EXTRACCIÓN DE EMBEDDINGS ALTA RESOLUCIÓN ===")

# 1. Cargar SigLIP 384px (El "Ojo de Águila")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Cargando modelo pesado SigLIP SO400M (Esto tardará 1 minuto)...")

# open_clip nos da el modelo y el preprocesador que redimensiona a 384x384
model, _, preprocess = open_clip.create_model_and_transforms('ViT-SO400M-14-SigLIP2-378', pretrained='webli')

if torch.cuda.device_count() > 1:
    print(f"🔥 ¡Activando DataParallel en {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)
model = model.to(device)
model.eval()

# 2. Clase Dataset Universal
class HighResImageDataset(Dataset):
    def __init__(self, image_paths, ids, preprocess):
        self.image_paths = image_paths
        self.ids = ids
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        item_id = self.ids[idx]
        try:
            img = Image.open(path).convert("RGB")
            img_tensor = self.preprocess(img)
            return img_tensor, item_id
        except Exception:
            # Fallback en caso de imagen corrupta
            return torch.zeros((3, 384, 384)), item_id

# 3. Procesar Productos
valid_products = products_df[products_df['product_asset_id'].apply(lambda x: os.path.exists(PRODUCTS_IMG_DIR / f"{x}.jpg"))]
p_paths = [PRODUCTS_IMG_DIR / f"{pid}.jpg" for pid in valid_products['product_asset_id']]
p_ids = valid_products['product_asset_id'].tolist()

prod_dataset = HighResImageDataset(p_paths, p_ids, preprocess)
# Usamos un batch size un poco más bajo (64) porque la resolución 384 ocupa mucha VRAM
prod_loader = DataLoader(prod_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

product_embeddings = {}
with torch.no_grad():
    for images, ids in tqdm(prod_loader, desc="Inferencia Productos (SigLIP)"):
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            # El método encode_image es el estándar en open_clip
            features = model.module.encode_image(images) if isinstance(model, torch.nn.DataParallel) else model.encode_image(images)
            
        # Normalización L2 imprescindible para el producto punto posterior
        features = features / features.norm(dim=-1, keepdim=True)
        features = features.cpu().float().numpy()
        
        for i, p_id in enumerate(ids):
            product_embeddings[p_id] = features[i]

# 4. Procesar Recortes de Bundles
crop_files = list(CROPS_DIR.glob("*.jpg"))
c_paths = [str(p) for p in crop_files]
# c_ids tendrá formato "B_1234_HEAD", "B_1234_GLOBAL", etc.
c_ids = [p.stem for p in crop_files] 

crop_dataset = HighResImageDataset(c_paths, c_ids, preprocess)
crop_loader = DataLoader(crop_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

bundle_crop_embeddings = {}
with torch.no_grad():
    for images, ids in tqdm(crop_loader, desc="Inferencia Crops Anatómicos"):
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            features = model.module.encode_image(images) if isinstance(model, torch.nn.DataParallel) else model.encode_image(images)
            
        features = features / features.norm(dim=-1, keepdim=True)
        features = features.cpu().float().numpy()
        
        for i, crop_id in enumerate(ids):
            # Parseamos el ID (ej: "B_c848ff0bf533_FEET" -> b_id="B_c848ff0bf533", zone="FEET")
            parts = crop_id.rsplit('_', 1)
            if len(parts) == 2:
                b_id, zone = parts
                if b_id not in bundle_crop_embeddings:
                    bundle_crop_embeddings[b_id] = {}
                bundle_crop_embeddings[b_id][zone] = features[i]

print(f"\nExtracción completada.")
print(f"Vectores de Productos: {len(product_embeddings)}")
print(f"Bundles con Multi-Vectores: {len(bundle_crop_embeddings)}")

=== 3. EXTRACCIÓN DE EMBEDDINGS ALTA RESOLUCIÓN ===
Cargando modelo pesado SigLIP SO400M (Esto tardará 1 minuto)...


open_clip_model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

🔥 ¡Activando DataParallel en 2 GPUs!


Inferencia Productos (SigLIP):   0%|          | 0/407 [00:00<?, ?it/s]

Inferencia Crops Anatómicos:   0%|          | 0/182 [00:00<?, ?it/s]


Extracción completada.
Vectores de Productos: 26039
Bundles con Multi-Vectores: 2331


In [11]:
import pickle



# Nombre del archivo basado en tu especificación

embeddings_filename = "embeddings_55.pkl"



print(f"--- Guardando embeddings en {embeddings_filename} ---")



# Consolidamos ambos diccionarios en uno solo para el "bloque 3.5"

embeddings_55 = {

    'product_embeddings': product_embeddings,

    'bundle_crop_embeddings': bundle_crop_embeddings,

    'metadata': {

        'model': 'dinov2_vitl14',

        'dim': '384',

        'preprocess': '384x384_bicubic'

    }

}



try:

    with open(embeddings_filename, 'wb') as f:

        pickle.dump(embeddings_55, f, protocol=pickle.HIGHEST_PROTOCOL)

    

    file_size = os.path.getsize(embeddings_filename) / (1024 * 1024)

    print(f"✅ Embeddings guardados exitosamente.")

    print(f"   - Archivo: {embeddings_filename}")

    print(f"   - Tamaño: {file_size:.2f} MB")

    print(f"   - Productos: {len(embeddings_55['product_embeddings'])}")

    print(f"   - Bundles: {len(embeddings_55['bundle_crop_embeddings'])}")

except Exception as e:

    print(f"❌ Error al guardar los embeddings: {e}")



# Opcional: Liberar memoria de los diccionarios locales si ya no se necesitan

# del dino_product_embeddings

# del dino_bundle_crop_embeddings

--- Guardando embeddings en embeddings_55.pkl ---
✅ Embeddings guardados exitosamente.
   - Archivo: embeddings_55.pkl
   - Tamaño: 167.12 MB
   - Productos: 26039
   - Bundles: 2331


## 🦖 Bloque 3.5: DINOv2 ViT-L/14 Embeddings (Ensemble Branch)

Extraemos embeddings de **DINOv2 ViT-L/14** (1024 dims) como segunda rama del ensemble.
DINOv2 captura features espaciales y estructurales complementarias a SigLIP, mejorando la cobertura de matching.

In [12]:
# ==========================================
# BLOQUE 3.5: DINOv2 ViT-L/14 EMBEDDINGS (ENSEMBLE BRANCH)
# ==========================================
import torch
import timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
import os
import numpy as np

print("=== 3.5 EXTRACCIÓN DE EMBEDDINGS DINOv2 ===")

# 1. Cargar DINOv2 ViT-L/14
print("Cargando DINOv2 ViT-L/14...")
dino_model = timm.create_model('vit_large_patch14_dinov2.lvd142m', pretrained=True, num_classes=0)

if torch.cuda.device_count() > 1:
    print(f"🔥 DataParallel en {torch.cuda.device_count()} GPUs para DINOv2")
    dino_model = torch.nn.DataParallel(dino_model)
dino_model = dino_model.to(device)
dino_model.eval()

# 2. Preprocesamiento para DINOv2 (518x518 es la resolución nativa de este modelo)
DINO_IMG_SIZE = 518
dino_preprocess = transforms.Compose([
    transforms.Resize((DINO_IMG_SIZE, DINO_IMG_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class DINOImageDataset(Dataset):
    def __init__(self, image_paths, ids, transform):
        self.image_paths = image_paths
        self.ids = ids
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        item_id = self.ids[idx]
        try:
            img = Image.open(path).convert("RGB")
            img_tensor = self.transform(img)
            return img_tensor, item_id
        except Exception:
            return torch.zeros((3, DINO_IMG_SIZE, DINO_IMG_SIZE)), item_id

# 3. Productos DINOv2
valid_products_dino = products_df[products_df['product_asset_id'].apply(
    lambda x: os.path.exists(PRODUCTS_IMG_DIR / f"{x}.jpg"))]
p_paths_d = [PRODUCTS_IMG_DIR / f"{pid}.jpg" for pid in valid_products_dino['product_asset_id']]
p_ids_d = valid_products_dino['product_asset_id'].tolist()

dino_prod_dataset = DINOImageDataset(p_paths_d, p_ids_d, dino_preprocess)
dino_prod_loader = DataLoader(dino_prod_dataset, batch_size=48, shuffle=False, num_workers=4, pin_memory=True)

dino_product_embeddings = {}
with torch.no_grad():
    for images, ids in tqdm(dino_prod_loader, desc="DINOv2 Productos"):
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            features = dino_model.module.forward_features(images) if isinstance(dino_model, torch.nn.DataParallel) else dino_model.forward_features(images)
            # Use CLS token
            if features.dim() == 3:
                features = features[:, 0]
        features = features / features.norm(dim=-1, keepdim=True)
        features = features.cpu().float().numpy()
        for i, p_id in enumerate(ids):
            dino_product_embeddings[p_id] = features[i]

DINO_DIM = list(dino_product_embeddings.values())[0].shape[0]
print(f"DINOv2 embedding dim: {DINO_DIM}")

# 4. Crops DINOv2
crop_files_d = list(CROPS_DIR.glob("*.jpg"))
c_paths_d = [str(p) for p in crop_files_d]
c_ids_d = [p.stem for p in crop_files_d]

dino_crop_dataset = DINOImageDataset(c_paths_d, c_ids_d, dino_preprocess)
dino_crop_loader = DataLoader(dino_crop_dataset, batch_size=48, shuffle=False, num_workers=4, pin_memory=True)

dino_bundle_crop_embeddings = {}
with torch.no_grad():
    for images, ids in tqdm(dino_crop_loader, desc="DINOv2 Crops"):
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            features = dino_model.module.forward_features(images) if isinstance(dino_model, torch.nn.DataParallel) else dino_model.forward_features(images)
            if features.dim() == 3:
                features = features[:, 0]
        features = features / features.norm(dim=-1, keepdim=True)
        features = features.cpu().float().numpy()
        for i, crop_id in enumerate(ids):
            parts = crop_id.rsplit('_', 1)
            if len(parts) == 2:
                b_id, zone = parts
                if b_id not in dino_bundle_crop_embeddings:
                    dino_bundle_crop_embeddings[b_id] = {}
                dino_bundle_crop_embeddings[b_id][zone] = features[i]

# Liberar VRAM de DINOv2 base
del dino_model
torch.cuda.empty_cache()

print(f"\n✅ DINOv2 Embeddings extraídos.")
print(f"Productos DINOv2: {len(dino_product_embeddings)}")
print(f"Bundles DINOv2: {len(dino_bundle_crop_embeddings)}")

=== 3.5 EXTRACCIÓN DE EMBEDDINGS DINOv2 ===
Cargando DINOv2 ViT-L/14...


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

🔥 DataParallel en 2 GPUs para DINOv2


DINOv2 Productos:   0%|          | 0/543 [00:00<?, ?it/s]

DINOv2 embedding dim: 1024


DINOv2 Crops:   0%|          | 0/243 [00:00<?, ?it/s]


✅ DINOv2 Embeddings extraídos.
Productos DINOv2: 26039
Bundles DINOv2: 2331


## 🧠 Bloque 4: Contrastive Fine-Tuning (K-Fold)

SigLIP extrae vectores de **1152 dimensiones** de altísima calidad, pero su espacio latente es genérico. 
Entrenamos una **Projection Head** (Red Neuronal Residual) usando Contrastive Learning (InfoNCE Loss) para adaptar estos vectores a la lógica de *outfits* de Zara.
- **Entrenamiento:** Usamos la vista `GLOBAL` del bundle frente a la foto del producto.
- **Validación:** 5-Fold Cross Validation para evitar el sobreajuste y quedarnos con los pesos matemáticamente perfectos.
- **Reproyección:** Pasamos *todos* los recortes de YOLO y los productos por esta red para crear el espacio vectorial definitivo.

In [46]:
# ==========================================
# BLOQUE 4.1: FINE-TUNING MEJORADO — DEEPER HEAD + COSINE LR + BIGGER BATCHES
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from sklearn.model_selection import KFold
import copy
 
print("=== 4.1 FINE-TUNING MEJORADO ===")
 
# ── get_ideal_zones (sin cambios) ──────────────────────────────────────────
def get_ideal_zones(desc):
    if not isinstance(desc, str):
        return ['GLOBAL']
    desc = desc.upper()
    if desc in {'HAT/HEADBAND', 'GLASSES', 'SCARF', 'EARRINGS', 'NECKLACE',
                'HAIR ACCESSORY', 'BERET', 'CAP', 'HEADBAND'}:
        return ['HEAD']
    if desc in {'SHOES', 'MOCCASINS', 'SANDAL', 'SNEAKERS', 'FLAT SHOES',
                'HEELED SHOES', 'ANKLE BOOT', 'HEELED BOOT', 'TRAINERS',
                'RAIN BOOT', 'SOCKS', 'BOOT', 'LACE-UP SHOES', 'LOAFERS',
                'ESPADRILLES', 'CLOG', 'SLIPPERS', 'KNEE-HIGH BOOT',
                'PLATFORM SHOES', 'BALLET FLATS', 'ATHLETIC SHOES'}:
        return ['FEET']
    if desc in {'TROUSERS', 'JEANS', 'SKIRT', 'BERMUDAS', 'SHORTS',
                'LEGGINGS', 'CARGO PANTS', 'WIDE LEG TROUSERS', 'CULOTTES',
                'JOGGERS', 'CHINOS', 'PALAZZO', 'CAPRI',
                'MINI SKIRT', 'MIDI SKIRT', 'LONG SKIRT', 'SKORT',
                'PANTALON', 'CROPPED TROUSERS'}:
        return ['LEGS']
    if desc in {'SHIRT', 'T-SHIRT', 'SWEATER', 'SWEATSHIRT', 'JACKET',
                'PUFFER', 'WAISTCOAT', 'TOP', 'BODY', 'BLOUSE',
                'POLO SHIRT', 'CARDIGAN', 'HOODIE', 'TANK TOP',
                'CROP TOP', 'VEST', 'CAMISOLE', 'BUSTIER',
                'CORSET', 'TUBE TOP', 'HENLEY', 'TURTLENECK',
                'OVERSHIRT', 'BOMBER', 'BLAZER', 'GILET',
                'WINDBREAKER', 'DENIM JACKET', 'LEATHER JACKET',
                'KNIT', 'TRICOT', 'JERSEY'}:
        return ['TORSO']
    if desc in {'DRESS', 'COAT', 'JUMPSUIT', 'OVERALL', 'DUNGAREES',
                'LONG DRESS', 'MIDI DRESS', 'MINI DRESS', 'SHIRT DRESS',
                'WRAP DRESS', 'KAFTAN', 'TUNIC', 'PONCHO', 'CAPE',
                'TRENCH', 'TRENCH COAT', 'LONG COAT', 'PYJAMA',
                'TRACKSUIT', 'SUIT', 'ROMPER', 'PLAYSUIT', 'KIMONO', 'ROBE'}:
        return ['TORSO', 'LEGS']
    if desc in {'HAND BAG-RUCKSACK', 'BAG', 'BELT', 'WATCH', 'BRACELET',
                'RING', 'BROOCH', 'TIE', 'POCKET SQUARE', 'KEYCHAIN',
                'PHONE CASE', 'SUNGLASSES', 'BACKPACK', 'CLUTCH',
                'CROSSBODY BAG', 'TOTE BAG', 'WALLET', 'FANNY PACK',
                'GLOVES', 'MITTENS', 'SWIMSUIT', 'BIKINI', 'SWIMMING TRUNKS'}:
        return ['GLOBAL']
    return ['GLOBAL']
 
def get_ideal_zone(desc):
    return get_ideal_zones(desc)[0]
 
# ── ★ Arquitectura más profunda, sin residual forzado ─────────────────────
class ProjectionHead(nn.Module):
    def __init__(self, embedding_dim=1152, hidden_dim=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.1),           # ★ menos dropout (era 0.2)
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, embedding_dim)
        )
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
 
    def forward(self, x):
        return F.normalize(x + self.net(x), p=2, dim=-1)
 
# ── Dataset (sin cambios) ──────────────────────────────────────────────────
class MultiZoneDataset(Dataset):
    def __init__(self, pairs, bundle_crop_dict, product_emb_dict,
                 product_desc_map, hard_negatives=None):
        self.b_dict = bundle_crop_dict
        self.p_dict = product_emb_dict
        self.desc_map = product_desc_map
        self.hard_negatives = hard_negatives
        self.expanded = []
        for b_id, p_id in pairs:
            desc   = self.desc_map.get(p_id, '')
            zones  = get_ideal_zones(desc)
            b_crops = self.b_dict.get(b_id, {})
            matched = False
            for zone in zones:
                if zone in b_crops:
                    self.expanded.append((b_id, p_id, zone))
                    matched = True
            if not matched:
                fallback = 'GLOBAL' if 'GLOBAL' in b_crops else (list(b_crops.keys())[0] if b_crops else None)
                if fallback:
                    self.expanded.append((b_id, p_id, fallback))
 
    def __len__(self):
        return len(self.expanded)
 
    def __getitem__(self, idx):
        b_id, p_id, zone = self.expanded[idx]
        b_emb = self.b_dict[b_id][zone]
        p_emb = self.p_dict[p_id]
        neg_emb = np.zeros_like(p_emb)
        if self.hard_negatives and (b_id, p_id) in self.hard_negatives:
            neg_ids = self.hard_negatives[(b_id, p_id)]
            if neg_ids:
                neg_id = neg_ids[np.random.randint(len(neg_ids))]
                if neg_id in self.p_dict:
                    neg_emb = self.p_dict[neg_id]
        return (torch.tensor(b_emb,   dtype=torch.float32),
                torch.tensor(p_emb,   dtype=torch.float32),
                torch.tensor(neg_emb, dtype=torch.float32))
 
# ── ★ Hard negative mining más profundo (top-200, guardar 20) ─────────────
print("Mining hard negatives...")
valid_pairs = np.array([
    (row['bundle_asset_id'], row['product_asset_id'])
    for _, row in train_df.iterrows()
    if row['bundle_asset_id'] in bundle_crop_embeddings
    and row['product_asset_id'] in product_embeddings
])
 
bundle_positives = defaultdict(set)
for b_id, p_id in valid_pairs:
    bundle_positives[b_id].add(p_id)
 
all_p_ids_list  = list(product_embeddings.keys())
all_p_matrix_np = np.vstack([product_embeddings[pid] for pid in all_p_ids_list])
all_p_matrix_np = all_p_matrix_np / (np.linalg.norm(all_p_matrix_np, axis=1, keepdims=True) + 1e-8)
 
hard_negatives = {}
for b_id in set(valid_pairs[:, 0]):
    if b_id not in bundle_crop_embeddings:
        continue
    b_global = bundle_crop_embeddings[b_id].get('GLOBAL')
    if b_global is None:
        continue
    b_norm = b_global / (np.linalg.norm(b_global) + 1e-8)
    sims   = all_p_matrix_np @ b_norm
    top_idx = np.argsort(sims)[::-1][:200]          # ★ era 50
    positives = bundle_positives[b_id]
    neg_ids = [all_p_ids_list[i] for i in top_idx if all_p_ids_list[i] not in positives]
    for p_id in positives:
        if p_id in product_embeddings:
            hard_negatives[(b_id, p_id)] = neg_ids[:20]  # ★ era 10
 
print(f"Hard negatives mined para {len(hard_negatives)} pares")
 
# ── ★ Loss: InfoNCE + Triplet con margin mayor ────────────────────────────
triplet_loss_fn = nn.TripletMarginLoss(margin=0.5, p=2)  # ★ era 0.3
 
def combined_loss(model, bundle_feats, product_feats, neg_feats):
    logit_scale = model.logit_scale.exp().clamp(max=100)
    logits      = logit_scale * torch.matmul(bundle_feats, product_feats.T)
    labels      = torch.arange(len(bundle_feats)).to(bundle_feats.device)
    loss_nce    = F.cross_entropy(logits, labels)
    neg_mask    = neg_feats.abs().sum(dim=-1) > 0
    if neg_mask.sum() > 0:
        loss_triplet = triplet_loss_fn(
            bundle_feats[neg_mask],
            product_feats[neg_mask],
            neg_feats[neg_mask]
        )
    else:
        loss_triplet = torch.tensor(0.0, device=bundle_feats.device)
    return loss_nce + 0.5 * loss_triplet
 
# ── ★ Training con batch_size=1024 + cosine LR + patience=5 ──────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
kf     = KFold(n_splits=5, shuffle=True, random_state=42)
 
best_val_loss_global = float('inf')
best_model_state     = None
best_fold            = -1
EPOCHS               = 30          # ★ era 20
BATCH_SIZE           = 1024        # ★ era 512 — más negativos en-batch para InfoNCE
 
all_fold_states = []  # ★ guardar todos los folds para ensemble
 
for fold, (train_idx, val_idx) in enumerate(kf.split(valid_pairs)):
    print(f"\n--- FOLD {fold + 1}/5 ---")
 
    train_ds = MultiZoneDataset(valid_pairs[train_idx], bundle_crop_embeddings,
                                 product_embeddings, product_desc_map,
                                 hard_negatives=hard_negatives)
    val_ds   = MultiZoneDataset(valid_pairs[val_idx],   bundle_crop_embeddings,
                                 product_embeddings, product_desc_map)
 
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
 
    model_ft  = ProjectionHead().to(device)
    optimizer = optim.AdamW(model_ft.parameters(), lr=3e-4, weight_decay=1e-3)  # ★ lr 3e-4
 
    # ★ Cosine annealing con warmup manual
    total_steps   = EPOCHS * len(train_loader)
    warmup_steps  = 2 * len(train_loader)   # 2 épocas de warmup
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-4,
        total_steps=total_steps,
        pct_start=warmup_steps/total_steps,
        anneal_strategy='cos'
    )
 
    best_val_fold   = float('inf')
    best_fold_state = None
    patience        = 5    # ★ era 3
    patience_ctr    = 0
 
    for epoch in range(EPOCHS):
        model_ft.train()
        train_loss = 0
        for b_emb, p_emb, neg_emb in train_loader:
            b_emb, p_emb, neg_emb = b_emb.to(device), p_emb.to(device), neg_emb.to(device)
            optimizer.zero_grad()
            loss = combined_loss(model_ft, model_ft(b_emb), model_ft(p_emb), model_ft(neg_emb))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_ft.parameters(), 1.0)  # ★ gradient clipping
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
 
        model_ft.eval()
        val_loss = 0
        with torch.no_grad():
            for b_emb, p_emb, _ in val_loader:
                b_emb, p_emb = b_emb.to(device), p_emb.to(device)
                logit_scale  = model_ft.logit_scale.exp().clamp(max=100)
                logits = logit_scale * torch.matmul(model_ft(b_emb), model_ft(p_emb).T)
                labels = torch.arange(len(b_emb)).to(device)
                val_loss += F.cross_entropy(logits, labels).item()
 
        avg_train = train_loss / max(len(train_loader), 1)
        avg_val   = val_loss   / max(len(val_loader),   1)
        print(f"  Epoch {epoch+1:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")
 
        if avg_val < best_val_fold and avg_val > 0:
            best_val_fold   = avg_val
            best_fold_state = copy.deepcopy(model_ft.state_dict())
            patience_ctr    = 0
        else:
            patience_ctr += 1
        if patience_ctr >= patience:
            print(f"  🛑 Early stopping en época {epoch+1}")
            break
 
    all_fold_states.append((best_val_fold, best_fold_state))
 
    if best_val_fold < best_val_loss_global:
        best_val_loss_global = best_val_fold
        best_model_state     = copy.deepcopy(best_fold_state)
        best_fold            = fold + 1
 
print(f"\n🏆 Mejor modelo: Fold {best_fold} | Val Loss: {best_val_loss_global:.4f}")
 
# ── ★ Reproyectar con ENSEMBLE de los 3 mejores folds ────────────────────
print("\n--- RE-PROYECTANDO CON ENSEMBLE DE 3 MEJORES FOLDS ---")
 
# Ordenar folds por val_loss, coger top-3
top3_states = [s for _, s in sorted(all_fold_states, key=lambda x: x[0])[:3]]
 
def project_with_ensemble(top3_states, bundle_crop_embeddings, product_embeddings, device):
    """Promedia las proyecciones de los 3 mejores folds."""
    all_bundle_projs  = []
    all_product_projs = []
 
    p_ids_list  = list(product_embeddings.keys())
    p_matrix    = torch.tensor(
        np.vstack([product_embeddings[pid] for pid in p_ids_list]),
        dtype=torch.float32
    ).to(device)
 
    for state in top3_states:
        m = ProjectionHead().to(device)
        m.load_state_dict(state)
        m.eval()
 
        # Bundles
        b_proj = {}
        with torch.no_grad():
            for b_id, zones_dict in bundle_crop_embeddings.items():
                b_proj[b_id] = {}
                for zone, emb in zones_dict.items():
                    t = torch.tensor(emb, dtype=torch.float32).unsqueeze(0).to(device)
                    b_proj[b_id][zone] = m(t).squeeze(0).cpu().numpy()
 
        # Products en batches
        batch_size = 2048
        p_projs = []
        with torch.no_grad():
            for i in range(0, len(p_matrix), batch_size):
                p_projs.append(m(p_matrix[i:i+batch_size]).cpu().numpy())
        p_proj_matrix = np.vstack(p_projs)
 
        all_bundle_projs.append(b_proj)
        all_product_projs.append(p_proj_matrix)
 
    # Promediar y renormalizar
    projected_bundle_crops = {}
    for b_id in bundle_crop_embeddings.keys():
        projected_bundle_crops[b_id] = {}
        for zone in bundle_crop_embeddings[b_id].keys():
            avg = np.mean([bp[b_id][zone] for bp in all_bundle_projs], axis=0)
            projected_bundle_crops[b_id][zone] = avg / (np.linalg.norm(avg) + 1e-8)
 
    avg_p_matrix = np.mean(all_product_projs, axis=0)
    # Renormalizar por fila
    norms = np.linalg.norm(avg_p_matrix, axis=1, keepdims=True)
    avg_p_matrix = avg_p_matrix / (norms + 1e-8)
 
    projected_products = {pid: avg_p_matrix[i] for i, pid in enumerate(p_ids_list)}
 
    return projected_bundle_crops, projected_products
 
projected_bundle_crops, projected_products = project_with_ensemble(
    top3_states, bundle_crop_embeddings, product_embeddings, device
)
 
print("✅ Fine-Tuning completado. Ensemble de 3 folds proyectado y listo.")

=== 4.1 FINE-TUNING MEJORADO ===
Mining hard negatives...
Hard negatives mined para 6463 pares

--- FOLD 1/5 ---
  Epoch 01 | Train: 5.9863 | Val: 3.8289
  Epoch 02 | Train: 4.4119 | Val: 3.2396
  Epoch 03 | Train: 3.4921 | Val: 2.8505
  Epoch 04 | Train: 2.9056 | Val: 2.5949
  Epoch 05 | Train: 2.5170 | Val: 2.4619
  Epoch 06 | Train: 2.2237 | Val: 2.3701
  Epoch 07 | Train: 1.9880 | Val: 2.3042
  Epoch 08 | Train: 1.7812 | Val: 2.2739
  Epoch 09 | Train: 1.6192 | Val: 2.2471
  Epoch 10 | Train: 1.4855 | Val: 2.2430
  Epoch 11 | Train: 1.3511 | Val: 2.2245
  Epoch 12 | Train: 1.2245 | Val: 2.2434
  Epoch 13 | Train: 1.1462 | Val: 2.2533
  Epoch 14 | Train: 1.0600 | Val: 2.2558
  Epoch 15 | Train: 0.9951 | Val: 2.2589
  Epoch 16 | Train: 0.9357 | Val: 2.2815
  🛑 Early stopping en época 16

--- FOLD 2/5 ---
  Epoch 01 | Train: 6.0797 | Val: 3.8453
  Epoch 02 | Train: 4.4097 | Val: 3.2399
  Epoch 03 | Train: 3.4939 | Val: 2.8308
  Epoch 04 | Train: 2.9319 | Val: 2.6034
  Epoch 05 | Train

In [47]:
# ==========================================
# BLOQUE 4.2: DINOv2 FINE-TUNING (ENSEMBLE BRANCH)
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from sklearn.model_selection import KFold
import copy

print("=== 4.2 FINE-TUNING DINOv2 PROJECTION HEAD ===")

# 1. ProjectionHead adapted for DINOv2 dim
class DinoProjectionHead(nn.Module):
    def __init__(self, embedding_dim=None, hidden_dim=1536):
        super().__init__()
        if embedding_dim is None:
            embedding_dim = DINO_DIM
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, embedding_dim)
        )
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        
    def forward(self, x):
        projected = x + self.net(x)
        return F.normalize(projected, p=2, dim=-1)

# 2. Reuse MultiZoneDataset with DINOv2 embeddings
dino_valid_pairs = np.array([(row['bundle_asset_id'], row['product_asset_id']) 
                              for _, row in train_df.iterrows() 
                              if row['bundle_asset_id'] in dino_bundle_crop_embeddings 
                              and row['product_asset_id'] in dino_product_embeddings])

# Build hard negatives for DINOv2
dino_bundle_positives = defaultdict(set)
for b_id, p_id in dino_valid_pairs:
    dino_bundle_positives[b_id].add(p_id)

dino_all_p_ids = list(dino_product_embeddings.keys())
dino_all_p_matrix = np.vstack([dino_product_embeddings[pid] for pid in dino_all_p_ids])
dino_all_p_matrix = dino_all_p_matrix / (np.linalg.norm(dino_all_p_matrix, axis=1, keepdims=True) + 1e-8)

print("Mining DINOv2 hard negatives...")
dino_hard_negatives = {}
for b_id in set(dino_valid_pairs[:, 0]):
    if b_id not in dino_bundle_crop_embeddings:
        continue
    b_global = dino_bundle_crop_embeddings[b_id].get('GLOBAL')
    if b_global is None:
        continue
    b_norm = b_global / (np.linalg.norm(b_global) + 1e-8)
    sims = dino_all_p_matrix @ b_norm
    top50_idx = np.argsort(sims)[::-1][:50]
    positives = dino_bundle_positives[b_id]
    neg_ids = [dino_all_p_ids[i] for i in top50_idx if dino_all_p_ids[i] not in positives]
    for p_id in positives:
        if p_id in dino_product_embeddings:
            dino_hard_negatives[(b_id, p_id)] = neg_ids[:10]

print(f"DINOv2 hard negatives: {len(dino_hard_negatives)} pairs")

# 3. Training
dino_triplet_loss_fn = nn.TripletMarginLoss(margin=0.3, p=2)

def dino_combined_loss(model, bundle_feats, product_feats, neg_feats):
    logit_scale = model.logit_scale.exp()
    logits = logit_scale * torch.matmul(bundle_feats, product_feats.T)
    labels = torch.arange(len(bundle_feats)).to(bundle_feats.device)
    loss_nce = F.cross_entropy(logits, labels)
    neg_mask = neg_feats.abs().sum(dim=-1) > 0
    if neg_mask.sum() > 0:
        loss_triplet = dino_triplet_loss_fn(bundle_feats[neg_mask], product_feats[neg_mask], neg_feats[neg_mask])
    else:
        loss_triplet = torch.tensor(0.0, device=bundle_feats.device)
    return loss_nce + 0.5 * loss_triplet

kf_dino = KFold(n_splits=5, shuffle=True, random_state=42)
best_dino_val_loss = float('inf')
best_dino_state = None
best_dino_fold = -1

for fold, (train_idx, val_idx) in enumerate(kf_dino.split(dino_valid_pairs)):
    print(f"\n--- DINOv2 FOLD {fold + 1}/5 ---")
    
    train_ds = MultiZoneDataset(dino_valid_pairs[train_idx], dino_bundle_crop_embeddings,
                                 dino_product_embeddings, product_desc_map, dino_hard_negatives)
    val_ds = MultiZoneDataset(dino_valid_pairs[val_idx], dino_bundle_crop_embeddings,
                               dino_product_embeddings, product_desc_map, None)
    
    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, drop_last=True)
    
    dino_head = DinoProjectionHead().to(device)
    optimizer = optim.AdamW(dino_head.parameters(), lr=5e-4, weight_decay=1e-3)
    
    best_fold_loss = float('inf')
    best_fold_weights = None
    patience_ctr = 0
    
    for epoch in range(20):
        dino_head.train()
        t_loss = 0
        for b_emb, p_emb, neg_emb in train_loader:
            b_emb, p_emb, neg_emb = b_emb.to(device), p_emb.to(device), neg_emb.to(device)
            optimizer.zero_grad()
            loss = dino_combined_loss(dino_head, dino_head(b_emb), dino_head(p_emb), dino_head(neg_emb))
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
        
        dino_head.eval()
        v_loss = 0
        with torch.no_grad():
            for b_emb, p_emb, _ in val_loader:
                b_emb, p_emb = b_emb.to(device), p_emb.to(device)
                logit_scale = dino_head.logit_scale.exp()
                logits = logit_scale * torch.matmul(dino_head(b_emb), dino_head(p_emb).T)
                labels = torch.arange(len(b_emb)).to(device)
                v_loss += F.cross_entropy(logits, labels).item()
        
        avg_t = t_loss / max(len(train_loader), 1)
        avg_v = v_loss / max(len(val_loader), 1)
        print(f"  Epoch {epoch+1:02d} | Train: {avg_t:.4f} | Val: {avg_v:.4f}")
        
        if avg_v < best_fold_loss and avg_v > 0:
            best_fold_loss = avg_v
            best_fold_weights = copy.deepcopy(dino_head.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
        if patience_ctr >= 3:
            print(f"  🛑 Early Stopping epoch {epoch+1}")
            break
    
    if best_fold_loss < best_dino_val_loss:
        best_dino_val_loss = best_fold_loss
        best_dino_state = copy.deepcopy(best_fold_weights)
        best_dino_fold = fold + 1

print(f"\n🏆 Mejor modelo DINOv2: Fold {best_dino_fold} con Val Loss: {best_dino_val_loss:.4f}")

# 4. Reproyectar DINOv2
print("\n--- RE-PROYECTANDO EMBEDDINGS DINOv2 ---")
final_dino = DinoProjectionHead().to(device)
final_dino.load_state_dict(best_dino_state)
final_dino.eval()

projected_dino_bundle_crops = {}
projected_dino_products = {}

with torch.no_grad():
    for b_id, zones_dict in dino_bundle_crop_embeddings.items():
        projected_dino_bundle_crops[b_id] = {}
        for zone, emb in zones_dict.items():
            t_emb = torch.tensor(emb, dtype=torch.float32).unsqueeze(0).to(device)
            projected_dino_bundle_crops[b_id][zone] = final_dino(t_emb).squeeze(0).cpu().numpy()
    
    d_ids = list(dino_product_embeddings.keys())
    d_matrix = torch.tensor(np.vstack([dino_product_embeddings[pid] for pid in d_ids]), dtype=torch.float32).to(device)
    
    new_d_matrix = []
    for i in range(0, len(d_matrix), 2048):
        batch = d_matrix[i:i+2048]
        new_d_matrix.append(final_dino(batch).cpu().numpy())
    new_d_matrix = np.vstack(new_d_matrix)
    for i, pid in enumerate(d_ids):
        projected_dino_products[pid] = new_d_matrix[i]

print("✅ DINOv2 Fine-Tuning completado. Vectores DINOv2 listos para ensemble.")

=== 4.2 FINE-TUNING DINOv2 PROJECTION HEAD ===
Mining DINOv2 hard negatives...
DINOv2 hard negatives: 6463 pairs

--- DINOv2 FOLD 1/5 ---
  Epoch 01 | Train: 4.9043 | Val: 3.9171
  Epoch 02 | Train: 3.7432 | Val: 3.4258
  Epoch 03 | Train: 3.1779 | Val: 3.1759
  Epoch 04 | Train: 2.8084 | Val: 3.0315
  Epoch 05 | Train: 2.5541 | Val: 2.9061
  Epoch 06 | Train: 2.3486 | Val: 2.8337
  Epoch 07 | Train: 2.1695 | Val: 2.7688
  Epoch 08 | Train: 2.0149 | Val: 2.7332
  Epoch 09 | Train: 1.8825 | Val: 2.7338
  Epoch 10 | Train: 1.7778 | Val: 2.7378
  Epoch 11 | Train: 1.6806 | Val: 2.7248
  Epoch 12 | Train: 1.5913 | Val: 2.7139
  Epoch 13 | Train: 1.5135 | Val: 2.7087
  Epoch 14 | Train: 1.4453 | Val: 2.7168
  Epoch 15 | Train: 1.3813 | Val: 2.7299
  Epoch 16 | Train: 1.3302 | Val: 2.7090
  🛑 Early Stopping epoch 16

--- DINOv2 FOLD 2/5 ---
  Epoch 01 | Train: 5.0272 | Val: 4.0919
  Epoch 02 | Train: 3.8849 | Val: 3.5104
  Epoch 03 | Train: 3.2797 | Val: 3.2673
  Epoch 04 | Train: 2.8823 | V

## 🏆 Bloque 5: The Beast Search Engine & Final Submission

Fase final de ensamblaje. Cruzamos el espacio latente afinado con la lógica humana:
1. **Smart Matching:** Los candidatos se evalúan contra su recorte anatómico ideal.
2. **Hard Constraints:** Filtrado estricto por reglas de sección (Bloque 1).
3. **Metadata Boost:** Bonificaciones por Timestamp, SKU Prefix y Co-ocurrencia de catálogo.
4. **SKU Deduplication:** Forzamos la diversidad en el Top 15 eliminando variantes de color redundantes.

In [48]:
# ==========================================
# BLOQUE 5: THE BEAST SEARCH ENGINE V12
# ==========================================
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import base64
from IPython.display import HTML
 
print("=== 5. MOTOR DE BÚSQUEDA V12: B2B_TEST_EXPANSION + SKU10 ===")
 
# 1. Metadatos
b_sku_map = bundles_df.set_index('bundle_asset_id')['sku'].to_dict()
b_ts_map  = bundles_df.set_index('bundle_asset_id')['ts'].to_dict()
p_sku_map = products_df.set_index('product_asset_id')['sku'].to_dict()
p_ts_map  = products_df.set_index('product_asset_id')['ts'].to_dict()
 
# 2. Co-ocurrencia base desde train
cooccurrence = defaultdict(Counter)
train_bids_grouped = train_df.groupby('bundle_asset_id')['product_asset_id'].apply(list)
for pids in train_bids_grouped:
    for p1 in pids:
        for p2 in pids:
            if p1 != p2:
                cooccurrence[p1][p2] += 1
 
# 3. Matrices de productos
all_p_ids = list(projected_products.keys())
siglip_p_matrix = np.vstack([projected_products[pid] for pid in all_p_ids])
pid_to_idx = {pid: i for i, pid in enumerate(all_p_ids)}
 
has_dino = 'projected_dino_products' in dir() and len(projected_dino_products) > 0
if has_dino:
    dino_p_matrix = np.vstack([projected_dino_products.get(pid, np.zeros(DINO_DIM)) for pid in all_p_ids])
    print(f"Ensemble activo: SigLIP ({siglip_p_matrix.shape[1]}d) + DINOv2 ({dino_p_matrix.shape[1]}d)")
else:
    print("Solo SigLIP")
 
# B2B matrix — solo train (base)
train_bundle_ids_b2b = [bid for bid in ground_truth_b2b.keys()
                         if bid in projected_bundle_crops and 'GLOBAL' in projected_bundle_crops[bid]]
train_bundle_matrix = np.vstack([projected_bundle_crops[bid]['GLOBAL'] for bid in train_bundle_ids_b2b])
print(f"B2B train: {len(train_bundle_ids_b2b)} bundles indexados.")
 
# ★ B2B ampliado — incluirá test bundles de alta confianza tras pasada 1
extended_b2b_ids    = list(train_bundle_ids_b2b)
extended_b2b_matrix = train_bundle_matrix.copy()
extended_ground_truth = dict(ground_truth_b2b)  # copia mutable
 
MAX_PER_CAT = {
    'SHIRT': 3, 'T-SHIRT': 3, 'TROUSERS': 3, 'JEANS': 3, 'DRESS': 3,
    'SWEATER': 3, 'JACKET': 3, 'SKIRT': 3, 'BLOUSE': 3, 'COAT': 3,
    'HAND BAG-RUCKSACK': 2, 'GLASSES': 2, 'HAT/HEADBAND': 2,
    'EARRINGS': 2, 'NECKLACE': 2, 'SCARF': 2, 'BELT': 2,
    'SHOES': 2, 'SNEAKERS': 2, 'SANDAL': 2, 'FLAT SHOES': 2,
    'HEELED SHOES': 2, 'ANKLE BOOT': 2, 'TRAINERS': 2, 'BOOT': 2,
}
DEFAULT_MAX_CAT = 5
 
ZONES_FOR_QUERY = ['GLOBAL', 'HEAD', 'TORSO', 'LEGS', 'FEET']
TOP_PER_ZONE = 2000
B2B_TOP_K    = 25
B2B_BOOST    = 0.20
 
test_bundles = test_df['bundle_asset_id'].unique()
 
def run_scoring_pass(test_bundles, cooccurrence, b2b_ids, b2b_matrix, b2b_gt):
    submission = []
    bundle_confidence = {}
    bundle_top15_map  = {}
 
    for b_id in test_bundles:
        if b_id not in projected_bundle_crops:
            continue
 
        b_crops_sig  = projected_bundle_crops[b_id]
        b_crops_dino = projected_dino_bundle_crops.get(b_id, {}) if has_dino else {}
        b_sku = str(b_sku_map.get(b_id, ''))
        b_ts  = b_ts_map.get(b_id)
        b_sec = bundle_section_map.get(b_id)
 
        candidate_set = set()
        zone_sims = {}
 
        for zone in ZONES_FOR_QUERY:
            if zone in b_crops_sig:
                sims = siglip_p_matrix @ b_crops_sig[zone]
                zone_sims[zone] = sims
                top_idx = np.argsort(sims)[::-1][:TOP_PER_ZONE]
                candidate_set.update(top_idx.tolist())
 
        if 'GLOBAL' not in zone_sims:
            fallback = list(b_crops_sig.values())[0]
            zone_sims['GLOBAL'] = siglip_p_matrix @ fallback
 
        dino_zone_sims = {}
        if has_dino:
            for zone in ZONES_FOR_QUERY:
                if zone in b_crops_dino:
                    dino_zone_sims[zone] = dino_p_matrix @ b_crops_dino[zone]
            if 'GLOBAL' not in dino_zone_sims and b_crops_dino:
                dino_zone_sims['GLOBAL'] = dino_p_matrix @ list(b_crops_dino.values())[0]
 
        # B2B con índice expandido
        b2b_product_boost = defaultdict(float)
        if 'GLOBAL' in b_crops_sig:
            b2b_sims = b2b_matrix @ b_crops_sig['GLOBAL']
            b2b_top  = np.argsort(b2b_sims)[::-1][:B2B_TOP_K]
            for rank, idx in enumerate(b2b_top):
                ref_bid = b2b_ids[idx]
                if ref_bid == b_id:
                    continue  # no compararse consigo mismo
                sim   = float(b2b_sims[idx])
                decay = 1.0 / (rank + 1)
                for prod in b2b_gt.get(ref_bid, []):
                    b2b_product_boost[prod] += sim * decay * B2B_BOOST
                    if prod in pid_to_idx:
                        candidate_set.add(pid_to_idx[prod])
 
        candidates = []
        for idx in candidate_set:
            p_id = all_p_ids[idx]
            desc = product_desc_map.get(p_id, '')
            if not is_valid_for_section(p_id, b_id):
                continue
 
            ideal_zones = get_ideal_zones(desc)
            ideal_sims  = [zone_sims[z][idx] for z in ideal_zones if z in zone_sims]
            sz = max(ideal_sims) if ideal_sims else zone_sims['GLOBAL'][idx]
            sg = zone_sims['GLOBAL'][idx]
            siglip_score = 0.80 * float(sz) + 0.20 * float(sg)
 
            if has_dino and dino_zone_sims:
                dino_ideal = [dino_zone_sims[z][idx] for z in ideal_zones if z in dino_zone_sims]
                dz = max(dino_ideal) if dino_ideal else (dino_zone_sims['GLOBAL'][idx] if 'GLOBAL' in dino_zone_sims else 0.0)
                dg = dino_zone_sims['GLOBAL'][idx] if 'GLOBAL' in dino_zone_sims else 0.0
                dino_score   = 0.70 * float(dz) + 0.30 * float(dg)
                visual_score = 0.85 * siglip_score + 0.15 * dino_score
            else:
                visual_score = siglip_score
 
            score = visual_score * 100
 
            p_sku = str(p_sku_map.get(p_id, ''))
            p_ts  = p_ts_map.get(p_id)
            is_accessory = desc in {
                'SHOES', 'SNEAKERS', 'HAT/HEADBAND', 'GLASSES', 'HAND BAG-RUCKSACK',
                'SANDAL', 'FLAT SHOES', 'HEELED SHOES', 'ANKLE BOOT', 'TRAINERS',
                'EARRINGS', 'NECKLACE', 'SCARF'
            }
 
            if b_ts and p_ts and not pd.isna(b_ts) and not pd.isna(p_ts):
                diff_days = abs(b_ts - p_ts) / (86400 * 1000)
                if diff_days < 7:    score *= 1.20
                elif diff_days < 14: score *= 1.14
                elif diff_days < 45: score *= 1.06
                elif diff_days > 180 and not is_accessory: score *= 0.88
 
            if b_sku != 'nan' and p_sku != 'nan' and len(b_sku) >= 6 and len(p_sku) >= 6:
                if b_sku[:6] == p_sku[:6]:
                    score *= 1.15
 
            sec_freq = section_product_freq[b_sec][p_id] if b_sec else 0
            if sec_freq > 0:
                score *= (1.0 + 0.04 * math.log1p(sec_freq))
 
            b2b_val = b2b_product_boost.get(p_id, 0.0)
            if b2b_val > 0:
                score *= (1.0 + min(b2b_val, 0.30))
 
            candidates.append({
                'product_id': p_id,
                'score':      score,
                # ★ sku_base con 10 chars en vez de 8
                'sku_base':   p_sku[:10] if p_sku != 'nan' else p_id
            })
 
        candidates.sort(key=lambda x: x['score'], reverse=True)
 
        if len(candidates) > 12:
            anchors = [c['product_id'] for c in candidates[:12]]
            for cand in candidates:
                cooc_bonus = 0
                for anchor in anchors:
                    if cand['product_id'] != anchor and cand['product_id'] in cooccurrence[anchor]:
                        cooc_bonus += min(cooccurrence[anchor][cand['product_id']], 3) * 0.02
                cand['score'] *= (1.0 + min(cooc_bonus, 0.30))
            candidates.sort(key=lambda x: x['score'], reverse=True)
 
        final_top_15 = []
        seen_skus    = set()
        seen_cats    = Counter()
 
        for cand in candidates:
            p_id     = cand['product_id']
            desc     = product_desc_map.get(p_id, '')
            sku_base = cand['sku_base']
            max_cat  = MAX_PER_CAT.get(desc, DEFAULT_MAX_CAT)
            if sku_base not in seen_skus and seen_cats[desc] < max_cat:
                final_top_15.append(p_id)
                seen_skus.add(sku_base)
                seen_cats[desc] += 1
            if len(final_top_15) == 15:
                break
 
        bundle_confidence[b_id]  = candidates[0]['score'] if candidates else 0
        bundle_top15_map[b_id]   = final_top_15
 
        for p_id in final_top_15:
            submission.append({'bundle_asset_id': b_id, 'product_asset_id': p_id})
 
    return submission, bundle_confidence, bundle_top15_map
 
def inject_self_training(bundle_confidence, bundle_top15_map, cooccurrence, percentile=75, weight=0.5):
    scores_arr = np.array(list(bundle_confidence.values()))
    threshold  = np.percentile(scores_arr, percentile)
    injected   = 0
    for b_id, conf in bundle_confidence.items():
        if conf >= threshold:
            pids = bundle_top15_map[b_id]
            for p1 in pids:
                for p2 in pids:
                    if p1 != p2:
                        cooccurrence[p1][p2] += weight
            injected += 1
    return injected, threshold
 
# ============================================================
# PASADA 1 — con B2B solo train
# ============================================================
print("Pasada 1/3...")
submission1, bundle_confidence1, bundle_top15_1 = run_scoring_pass(
    test_bundles, cooccurrence,
    extended_b2b_ids, extended_b2b_matrix, extended_ground_truth
)
injected, thr = inject_self_training(bundle_confidence1, bundle_top15_1, cooccurrence, percentile=75, weight=0.5)
print(f"Self-training 1: {injected} bundles inyectados (thr={thr:.2f})")
 
# ★ Expandir B2B con test bundles de alta confianza
print("Expandiendo B2B con test bundles de alta confianza...")
scores_arr = np.array(list(bundle_confidence1.values()))
conf_threshold = np.percentile(scores_arr, 80)
new_b2b_ids    = []
new_b2b_vecs   = []
new_b2b_gt     = {}
 
for b_id, conf in bundle_confidence1.items():
    if conf >= conf_threshold and b_id in projected_bundle_crops and 'GLOBAL' in projected_bundle_crops[b_id]:
        preds = bundle_top15_1[b_id]
        if preds:
            new_b2b_ids.append(b_id)
            new_b2b_vecs.append(projected_bundle_crops[b_id]['GLOBAL'])
            new_b2b_gt[b_id] = preds
 
if new_b2b_ids:
    extended_b2b_ids    = list(train_bundle_ids_b2b) + new_b2b_ids
    extended_b2b_matrix = np.vstack([train_bundle_matrix] + [np.array(v) for v in new_b2b_vecs])
    extended_ground_truth = {**ground_truth_b2b, **new_b2b_gt}
    print(f"  → {len(new_b2b_ids)} test bundles añadidos al índice B2B ({len(extended_b2b_ids)} total)")
 
# ============================================================
# PASADA 2 — con B2B expandido
# ============================================================
print("Pasada 2/3...")
submission2, bundle_confidence2, bundle_top15_2 = run_scoring_pass(
    test_bundles, cooccurrence,
    extended_b2b_ids, extended_b2b_matrix, extended_ground_truth
)
injected2, thr2 = inject_self_training(bundle_confidence2, bundle_top15_2, cooccurrence, percentile=85, weight=0.3)
print(f"Self-training 2: {injected2} bundles inyectados (thr={thr2:.2f})")
 
# ============================================================
# PASADA 3 — final con B2B expandido
# ============================================================
print("Pasada 3/3...")
submission3, _, _ = run_scoring_pass(
    test_bundles, cooccurrence,
    extended_b2b_ids, extended_b2b_matrix, extended_ground_truth
)
 
submission_df = pd.DataFrame(submission3)
file_path = WORK_DIR / 'submission_beast_v12.csv'
submission_df.to_csv(file_path, index=False)
print(f"\n✅ SUBMISSION V12 GENERADA! Total filas: {len(submission_df)}")
 
def create_download_link(file_path, title="📥 Descargar BEAST V12 Submission", filename="submission_beast_v12.csv"):
    with open(file_path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    html = f'''<a href="data:text/csv;base64,{b64}" download="{filename}"
               style="display: inline-block; padding: 15px 25px; background-color: #FF1493; color: white;
               text-decoration: none; font-size: 16px; border-radius: 8px; font-weight: bold;">{title}</a>'''
    return HTML(html)
 
display(create_download_link(file_path))

=== 5. MOTOR DE BÚSQUEDA V12: B2B_TEST_EXPANSION + SKU10 ===
Ensemble activo: SigLIP (1152d) + DINOv2 (1024d)
B2B train: 1876 bundles indexados.
Pasada 1/3...
Self-training 1: 114 bundles inyectados (thr=104.18)
Expandiendo B2B con test bundles de alta confianza...
  → 91 test bundles añadidos al índice B2B (1967 total)
Pasada 2/3...
Self-training 2: 69 bundles inyectados (thr=119.98)
Pasada 3/3...

✅ SUBMISSION V12 GENERADA! Total filas: 6825


In [ ]:
# ==========================================
# BLOQUE 6: EVALUACIÓN RECALL@15 EN TRAIN
# ==========================================
print("=== 6. EVALUACIÓN RECALL@15 EN TRAIN ===")

# Run the same pipeline on train bundles to compute Recall@15
train_bundles_eval = train_df['bundle_asset_id'].unique()

# Build ground truth: bundle_id -> set of product_ids
ground_truth = defaultdict(set)
for _, row in train_df.iterrows():
    ground_truth[row['bundle_asset_id']].add(row['product_asset_id'])

total_hits = 0
total_possible = 0

for b_id in train_bundles_eval:
    if b_id not in projected_bundle_crops:
        continue
    
    b_crops_sig = projected_bundle_crops[b_id]
    b_crops_dino = projected_dino_bundle_crops.get(b_id, {}) if has_dino else {}
    b_sku = str(b_sku_map.get(b_id, ''))
    b_ts = b_ts_map.get(b_id)
    
    # Multi-query fusion (SigLIP)
    candidate_set = set()
    zone_sims = {}
    
    for zone in ZONES_FOR_QUERY:
        if zone in b_crops_sig:
            sims = siglip_p_matrix @ b_crops_sig[zone]
            zone_sims[zone] = sims
            top_idx = np.argsort(sims)[::-1][:TOP_PER_ZONE]
            candidate_set.update(top_idx)
    
    if 'GLOBAL' not in zone_sims:
        fallback_emb = list(b_crops_sig.values())[0]
        zone_sims['GLOBAL'] = siglip_p_matrix @ fallback_emb
    
    # DINOv2 candidate retrieval + zone sims
    dino_zone_sims = {}
    if has_dino:
        for zone in ZONES_FOR_QUERY:
            if zone in b_crops_dino:
                dino_sims = dino_p_matrix @ b_crops_dino[zone]
                dino_zone_sims[zone] = dino_sims
                top_dino_idx = np.argsort(dino_sims)[::-1][:TOP_PER_ZONE]
                candidate_set.update(top_dino_idx)
        if 'GLOBAL' not in dino_zone_sims and b_crops_dino:
            dino_zone_sims['GLOBAL'] = dino_p_matrix @ list(b_crops_dino.values())[0]
    
    candidates = []
    for idx in candidate_set:
        p_id = all_p_ids[idx]
        desc = product_desc_map.get(p_id, '')
        
        if not is_valid_for_section(p_id, b_id):
            continue
        
        ideal_zones = get_ideal_zones(desc)
        ideal_sims = [zone_sims[z][idx] for z in ideal_zones if z in zone_sims]
        sig_zone_score = max(ideal_sims) if ideal_sims else zone_sims['GLOBAL'][idx]
        sig_global_score = zone_sims['GLOBAL'][idx]
        siglip_score = 0.6 * sig_zone_score + 0.4 * sig_global_score
        
        if has_dino and dino_zone_sims:
            dino_ideal = [dino_zone_sims[z][idx] for z in ideal_zones if z in dino_zone_sims]
            dino_z = max(dino_ideal) if dino_ideal else (dino_zone_sims['GLOBAL'][idx] if 'GLOBAL' in dino_zone_sims else 0.0)
            dino_g = dino_zone_sims['GLOBAL'][idx] if 'GLOBAL' in dino_zone_sims else 0.0
            dino_score = 0.6 * dino_z + 0.4 * dino_g
            visual_score = 0.6 * siglip_score + 0.4 * dino_score
        else:
            visual_score = siglip_score
        
        score = float(visual_score) * 100
        
        p_sku = str(p_sku_map.get(p_id, ''))
        p_ts = p_ts_map.get(p_id)
        is_acc = desc in {'SHOES', 'SNEAKERS', 'HAT/HEADBAND', 'GLASSES', 'HAND BAG-RUCKSACK',
                          'SANDAL', 'FLAT SHOES', 'HEELED SHOES', 'ANKLE BOOT', 'TRAINERS',
                          'EARRINGS', 'NECKLACE', 'SCARF'}
        
        if b_ts and p_ts and not pd.isna(b_ts) and not pd.isna(p_ts):
            diff_days = abs(b_ts - p_ts) / (86400 * 1000)
            if diff_days < 14: score *= 1.12
            elif diff_days < 45: score *= 1.05
            elif diff_days > 180 and not is_acc: score *= 0.92
        
        if b_sku != 'nan' and p_sku != 'nan' and len(b_sku) >= 6 and len(p_sku) >= 6:
            if b_sku[:6] == p_sku[:6]:
                score *= 1.15
        
        candidates.append({'product_id': p_id, 'score': score,
                          'sku_base': p_sku[:8] if p_sku != 'nan' else p_id})
    
    candidates.sort(key=lambda x: x['score'], reverse=True)
    
    if len(candidates) > 5:
        anchors = [c['product_id'] for c in candidates[:5]]
        for cand in candidates:
            cooc_bonus = 0
            for anchor in anchors:
                if cand['product_id'] != anchor and cand['product_id'] in cooccurrence[anchor]:
                    cooc_bonus += min(cooccurrence[anchor][cand['product_id']], 3) * 0.02
            cand['score'] *= (1.0 + min(cooc_bonus, 0.15))
    
    candidates.sort(key=lambda x: x['score'], reverse=True)
    
    # Deduplicate by SKU
    top_15 = []
    seen = set()
    for c in candidates:
        if c['sku_base'] not in seen:
            top_15.append(c['product_id'])
            seen.add(c['sku_base'])
        if len(top_15) == 15:
            break
    
    # Compute recall
    gt = ground_truth[b_id]
    hits = len(set(top_15) & gt)
    total_hits += hits
    total_possible += len(gt)

recall_at_15 = total_hits / total_possible if total_possible > 0 else 0
print(f"\n🎯 Recall@15 en Train: {recall_at_15:.4f} ({recall_at_15*100:.2f}%)")
print(f"   Hits: {total_hits} / {total_possible}")
print(f"⚠️  Nota: Co-ocurrencia evaluada con datos de train → métrica optimista.")